[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/26_lora.ipynb)

# 🟠 Medium: LoRA (Low-Rank Adaptation)

Implement **LoRA** — parameter-efficient fine-tuning for large models.

$$h = W_0 x + \frac{\alpha}{r} B A x$$

### Signature
```python
class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank, alpha=1.0): ...
    def forward(self, x: Tensor) -> Tensor: ...
```

### Requirements
- `self.linear`: frozen `nn.Linear` (weight & bias `requires_grad=False`)
- `self.lora_A`: `nn.Parameter(rank, in_features)` — random init
- `self.lora_B`: `nn.Parameter(out_features, rank)` — **zero** init
- Scaling: `alpha / rank`

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [1]:
import torch
import torch.nn as nn

In [2]:
# ✏️ YOUR IMPLEMENTATION HERE

class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank, alpha=1.0):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.linear.weight.requires_grad_(False)
        self.linear.bias.requires_grad_(False)
        self.A = nn.Parameter(torch.randn(rank, in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(out_features, rank))
        self.scaling = alpha / rank


    def forward(self, x):
        return self.linear(x) + (x @ self.A.T @ self.B.T) * self.scaling

In [3]:
# 🧪 Debug
layer = LoRALinear(16, 8, rank=4)
x = torch.randn(2, 16)
print('Output:', layer(x).shape)
print('Trainable:', sum(p.numel() for p in layer.parameters() if p.requires_grad))
print('Total:    ', sum(p.numel() for p in layer.parameters()))

Output: torch.Size([2, 8])
Trainable: 96
Total:     232


In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('lora')